# TestSafebound

## Build

### Init

In [1]:
import pandas as pd
import sys
import os

# ==================== 配置区域 ====================
# 数据目录配置：可以分别为 IMDB 和 Stats 指定不同的数据目录
# 默认使用 SafeBound 根目录下的 Data 目录
# 如果需要使用自定义数据目录，请修改下面的路径
default_root = os.path.abspath('../methods/SafeBound/')
if not default_root.endswith('/'):
    default_root += '/'

# IMDB 数据目录（指向包含 IMDB CSV 文件的目录，如 /path/to/Data/IMDB/）
imdb_data_directory = default_root + "Data/IMDB/"

# Stats 数据目录（指向包含 Stats CSV 文件的目录，如 /path/to/Data/Stats/）
stats_data_directory = default_root + "Data/Stats/"

# 示例：如果需要使用自定义路径，取消注释并修改下面的行：
# imdb_data_directory = "/path/to/your/imdb/data/directory/"
# stats_data_directory = "/path/to/your/stats/data/directory/"
imdb_data_directory = os.path.abspath("../methods/SafeBound/Data/IMDB")
stats_data_directory = os.path.abspath("../methods/SafeBound/Data/Stats")
# ================================================

# 获取 SafeBound 根目录（从 experiment/ 目录访问 methods/SafeBound/）
rootFileDirectory = os.path.abspath('../methods/SafeBound/')
if not rootFileDirectory.endswith('/'):
    rootFileDirectory += '/'

# 先添加 Source 目录（必须在最前面，这样 ExperimentUtils 才能作为包被导入）
# 注意：这必须在任何可能导入 experiment/ExperimentUtils.py 之前完成
source_path = rootFileDirectory + 'Source'
experiment_utils_path = rootFileDirectory + 'Source/ExperimentUtils'

# 如果 ExperimentUtils 已经被导入为文件模块，先清除它（避免命名冲突）
if 'ExperimentUtils' in sys.modules:
    # 检查是否是文件模块（不是包）
    module = sys.modules['ExperimentUtils']
    if hasattr(module, '__file__') and 'experiment' in module.__file__.lower():
        # 这是 experiment/ExperimentUtils.py 文件，需要删除
        del sys.modules['ExperimentUtils']

# 将 SafeBound 的 Source 目录添加到 sys.path 最前面（优先级最高）
if source_path not in sys.path:
    sys.path.insert(0, source_path)
if experiment_utils_path not in sys.path:
    sys.path.insert(1, experiment_utils_path)

# 在导入 BuildUtils 之前，先设置 LoadUtils 的数据目录
# 因为 ExperimentUtils 目录已经在 sys.path 中，可以直接导入 LoadUtils
import LoadUtils
LoadUtils.imdb_data_directory = imdb_data_directory
LoadUtils.stats_data_directory = stats_data_directory

# BuildUtils 也在 ExperimentUtils 目录中，可以直接导入
# （BuildUtils.py 内部已经添加了 Source 和 Source/ExperimentUtils 到 sys.path）
from BuildUtils import *

# 输出到 experiment/checkpoint/SafeBound 目录
checkpoint_dir = os.path.abspath('./checkpoint/SafeBound')
os.makedirs(checkpoint_dir, exist_ok=True)

def build_safebound_benchmark(benchmark):
    """
    构建 SafeBound 统计对象

    参数:
        benchmark: benchmark名称 ('Stats', 'JOBLight', 'JOBLightRanges', 'JOBM')

    返回:
        tuple: (构建时间, 统计对象大小)
    """
    # 直接使用第三个参数配置（索引为2）
    parameters = {
        'relativeErrorPerSegment': 0.02,
        'numHistogramBuckets': 32,
        'numEqualityOutliers': 512,
        'numCDFGroups': 16,
        'trackNulls': False,
        'trackTriGrams': False,
        'numCores': 18,
        'groupingMethod': "CompleteClustering",
        'modelCDF': True,
        'verbose': False
    }

    # JOBM的特殊处理
    if benchmark == 'JOBM':
        parameters['numEqualityOutliers'] = 5 * parameters['numEqualityOutliers']
        parameters['trackTriGrams'] = True
        parameters['trackNulls'] = True
        parameters['numCores'] = 6
        parameters['verbose'] = True

    outputFile = os.path.join(checkpoint_dir, f"SafeBound_3_{benchmark}.pkl")

    print(f"\n{'='*60}")
    print(f"处理 Benchmark: {benchmark}")
    print(f"使用参数配置：")
    for key, value in parameters.items():
        print(f"  {key}: {value}")
    print(f"输出文件: {outputFile}")

    # 构建统计对象
    time, size = build_stats_object(
        method='SafeBound',
        benchmark=benchmark,
        parameters=parameters,
        outputFile=outputFile
    )

    print(f"构建完成！")
    print(f"构建时间: {time:.2f} 秒")
    print(f"统计对象大小: {size} 字节")

    return time, size

print(f"IMDB 数据目录已配置为: {imdb_data_directory}")
print(f"Stats 数据目录已配置为: {stats_data_directory}")
print("配置完成，可以使用 build_safebound_benchmark() 函数构建各个 benchmark")

IMDB 数据目录已配置为: /home/liwei/starCE/methods/SafeBound/Data/IMDB
Stats 数据目录已配置为: /home/liwei/starCE/methods/SafeBound/Data/Stats
配置完成，可以使用 build_safebound_benchmark() 函数构建各个 benchmark


### Stats Benchmark

In [2]:
stats_time, stats_size = build_safebound_benchmark('Stats')
stats_time, stats_size


处理 Benchmark: Stats
使用参数配置：
  relativeErrorPerSegment: 0.02
  numHistogramBuckets: 32
  numEqualityOutliers: 512
  numCDFGroups: 16
  trackNulls: False
  trackTriGrams: False
  numCores: 18
  groupingMethod: CompleteClustering
  modelCDF: True
  verbose: False
输出文件: /home/liwei/starCE/experiment/checkpoint/SafeBound/SafeBound_3_Stats.pkl


/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

Table: BADGES
Filter Col: BADGES.DATE
Interval Footprint: 5440
Range Footprint: 4440.0
Equality Bloom Filter Footprint: 5154.0
Equality Outlier Footprint: 816.0
Equality Max Footprint: 48.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: USERS.REPUTATION
Interval Footprint: 5428
Range Footprint: 3504.0
Equality Bloom Filter Footprint: 4812.0
Equality Outlier Footprint: 1248.0
Equality Max Footprint: 48.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: USERS.CREATIONDATE
Interval Footprint: 5440
Range Footprint: 4536.0
Equality Bloom Filter Footprint: 4912.0
Equality Outlier Footprint: 768.0
Equality Max Footprint: 48.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: USERS.VIEWS
Interval Footprint: 5440
Range Footprint: 3552.0
Equality Bloom Filter Footprint: 4800.0
Equality Outlier Footprint: 1224.0
Equality Max Footprint: 48.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: USERS.UPVOTES
Interval Footprint: 5440
Range Footprint: 4128.0
Equality Bloom Filter Footprint: 4800

(18.091733, 1268042.0)

### JOBLight Benchmark

In [3]:
joblight_time, joblight_size = build_safebound_benchmark('JOBLight')
joblight_time, joblight_size


处理 Benchmark: JOBLight
使用参数配置：
  relativeErrorPerSegment: 0.02
  numHistogramBuckets: 32
  numEqualityOutliers: 512
  numCDFGroups: 16
  trackNulls: False
  trackTriGrams: False
  numCores: 18
  groupingMethod: CompleteClustering
  modelCDF: True
  verbose: False
输出文件: /home/liwei/starCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBLight.pkl


/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

Table: CAST_INFO
Filter Col: CAST_INFO.ROLE_ID
Interval Footprint: 1772
Range Footprint: 3960.0
Equality Bloom Filter Footprint: 3300.0
Equality Outlier Footprint: 1392.0
Equality Max Footprint: 24.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: CAST_INFO.NR_ORDER
Interval Footprint: 5440
Range Footprint: 2952.0
Equality Bloom Filter Footprint: 4852.0
Equality Outlier Footprint: 600.0
Equality Max Footprint: 24.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: TITLE.EPISODE_NR
Interval Footprint: 5428
Range Footprint: 3480.0
Equality Bloom Filter Footprint: 4902.0
Equality Outlier Footprint: 1752.0
Equality Max Footprint: 24.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: TITLE.SEASON_NR
Interval Footprint: 5428
Range Footprint: 3408.0
Equality Bloom Filter Footprint: 4800.0
Equality Outlier Footprint: 1152.0
Equality Max Footprint: 24.0
TriGram Footprint: 0
Null Footprint: 0
Filter Col: TITLE.KIND_ID
Interval Footprint: 1092
Range Footprint: 2688.0
Equality Bloom Filter F

(187.577585, 362156.0)

### JOBLightRanges Benchmark

In [ ]:
joblightranges_time, joblightranges_size = build_safebound_benchmark('JOBLightRanges')
joblightranges_time, joblightranges_size


处理 Benchmark: JOBLightRanges
使用参数配置：
  relativeErrorPerSegment: 0.02
  numHistogramBuckets: 32
  numEqualityOutliers: 512
  numCDFGroups: 16
  trackNulls: False
  trackTriGrams: False
  numCores: 18
  groupingMethod: CompleteClustering
  modelCDF: True
  verbose: False
输出文件: /home/liwei/starCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBLightRanges.pkl


### JOBM Benchmark

In [ ]:
jobm_time, jobm_size = build_safebound_benchmark('JOBM')
jobm_time, jobm_size


处理 Benchmark: JOBM
使用参数配置：
  relativeErrorPerSegment: 0.02
  numHistogramBuckets: 32
  numEqualityOutliers: 2560
  numCDFGroups: 16
  trackNulls: True
  trackTriGrams: True
  numCores: 6
  groupingMethod: CompleteClustering
  modelCDF: True
  verbose: True
输出文件: /home/liwei/starCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBM.pkl
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations
Building Full Table Approximations

/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)


Approximating null distributions: CAST_INFO.NOTE
Building Stats: TITLE.EPISODE_NR


/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)


Approximating null distributions: TITLE.EPISODE_NR
Approximating not null distributions: CAST_INFO.NOTE
Building Stats: TITLE.PRODUCTION_YEAR
Approximating not null distributions: TITLE.EPISODE_NR
Approximating range distributions: CAST_INFO.NOTE Memory Usage: 12.425908224


...

/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)


Approximating null distributions: TITLE.PRODUCTION_YEAR
Approximating not null distributions: TITLE.PRODUCTION_YEAR
Approximating range distributions: TITLE.PRODUCTION_YEAR Memory Usage: 15.515725824
Clustering Function Approximations For Most Common TriGrams: KEYWORD.KEYWORD 4523930 Memory Usage: 18.205282304
Outlier Group Sets: KEYWORD.KEYWORD 16
Handling Remainder Rows: KEYWORD.KEYWORD 4523930 Memory Usage: 18.205282304
Finished Building Stats: KEYWORD.KEYWORD
Building Stats: INFO_TYPE.INFO
Approximating equality distributions: INFO_TYPE.INFO Memory Usage: 11.134619648
Creating Function Approximations For Most Common TriGrams: MOVIE_INFO.INFO 14835720 Memory Usage: 23.622291456


/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/concurrent/futures/process.py:243: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r = call_item.fn(*call_item.args, **call_item.kwargs)


Approximating null distributions: TITLE.TITLE
Approximating not null distributions: TITLE.TITLE
Approximating equality distributions: TITLE.PRODUCTION_YEAR Memory Usage: 15.515725824
Finished Building Stats: TITLE.PRODUCTION_YEAR
Approximating range distributions: TITLE.TITLE Memory Usage: 17.751822336
Approximating null distributions: INFO_TYPE.INFO
Approximating not null distributions: INFO_TYPE.INFO
Approximating range distributions: INFO_TYPE.INFO Memory Usage: 15.57946368
Detecting Most Common TriGrams: INFO_TYPE.INFO 14835720 Memory Usage: 11.134619648
Approximating equality distributions: TITLE.TITLE Memory Usage: 17.753391104
Approximating equality distributions: INFO_TYPE.INFO Memory Usage: 15.57946368
Detecting Most Common TriGrams: INFO_TYPE.INFO 2963664 Memory Usage: 15.57946368
Creating Function Approximations For Most Common TriGrams: INFO_TYPE.INFO 2963664 Memory Usage: 15.77007104
Creating Function Approximations For Most Common TriGrams: INFO_TYPE.INFO 14835720 Memory 

(1893.179702, 3567828.0)

### StatsJoin Benchmark

In [ ]:
# StatsJoin 复用 Stats 的统计对象（同数据库、同 schema，仅为去谓词版本）
statsjoin_time, statsjoin_size = build_safebound_benchmark('Stats')
statsjoin_time, statsjoin_size

## Evaluate

### Init

In [ ]:
import pickle
import time
import json
import sys
import os
import re

# 添加 SafeBound Source 目录到路径
sys.path.append(rootFileDirectory + 'Source')
from SafeBoundUtils import *
from JoinGraphUtils import *
from SQLParser import *

def evaluate_safebound_benchmark(benchmark, sql_file_path, stat_object_path):
    """
    评估 SafeBound 统计对象对 SQL 查询的基数估计

    参数:
        benchmark: benchmark名称 ('Stats', 'JOBLight', 'JOBLightRanges', 'JOBM')
        sql_file_path: SQL查询文件路径
        stat_object_path: 统计对象 pickle 文件路径

    返回:
        dict: 包含评估结果的字典，包括：
            - total_time: 纯估计时间（秒），不含 SQL 解析
            - parse_time: SQL 解析时间（秒）
            - num_queries: 查询数量
            - results: 每个查询的估计结果列表
            - output_file: 结果输出文件路径
    """
    # 加载统计对象
    print(f"\n{'='*60}")
    print(f"加载统计对象: {stat_object_path}")
    safeBound = pickle.load(open(stat_object_path, 'rb'))
    print(f"统计对象大小: {os.path.getsize(stat_object_path)} 字节")

    # 读取 SQL 查询文件
    print(f"\n读取 SQL 查询文件: {sql_file_path}")
    with open(sql_file_path, 'r') as f:
        sqls = f.readlines()

    print(f"查询数量: {len(sqls)}")

    # 预解析所有 SQL 查询（解析不计入估计时间）
    print(f"\n预解析 {len(sqls)} 条 SQL 查询...")
    parse_start = time.time()
    parsed_entries = []  # 每个元素为 (line_id, sql, query) 或 None（解析失败）
    parse_failures = 0
    for line_id, sql in enumerate(sqls, 1):
        if (line_id - 1) % max(1, len(sqls) // 10) == 0 and line_id > 1:
            print(f"  预解析进度: {line_id - 1}/{len(sqls)}")
        try:
            if benchmark == 'Stats':
                sql_clean = sql.strip()
                if not sql_clean:  # 跳过空行
                    parsed_entries.append(None)
                    continue
                query = sql_to_joingraph(sql_clean, keep_type_cast=False)
            else:
                result = SQLQueriesToJoinQueryGraphs(sql)
                if len(result) > 0:
                    query = result[0]
                else:
                    print(f"警告: 查询 {line_id} 解析失败")
                    parsed_entries.append(None)
                    parse_failures += 1
                    continue
            query.buildJoinGraph()
            parsed_entries.append((line_id, sql, query))
        except Exception as e:
            print(f"警告: 查询 {line_id} 解析异常: {str(e)}")
            parsed_entries.append(None)
            parse_failures += 1
    parse_time = time.time() - parse_start
    successful_parses = len([e for e in parsed_entries if e is not None])
    print(f"预解析完成: 成功 {successful_parses}, 失败 {parse_failures}, 耗时 {parse_time:.2f}s（不计入估计时间）")

    # 计时：只测量基数估计时间（SQL 解析和文件写入不计入）
    results = []
    start_time = time.time()

    # 保存结果到 checkpoint 目录（每行一个估计值）
    output_file = os.path.join(checkpoint_dir, f"SafeBound_3_{benchmark}_evaluate_results.txt")

    with open(output_file, 'w') as f:
        latencies = []
        estimate_count = 0
        estimate_failures = 0
        for entry in parsed_entries:
            if entry is None:
                f.write('NaN\n')
                latencies.append(0.0)
                continue
            line_id, sql, query = entry
            try:
                # 执行基数估计
                _t0 = time.time()
                bound = safeBound.functionalFrequencyBound(query)
                latencies.append(time.time() - _t0)
                f.write(str(bound) + '\n')
                results.append({
                    'query_id': line_id,
                    'sql': sql,
                    'estimate': bound
                })

                estimate_count += 1
                if estimate_count % max(1, successful_parses // 10) == 0 or estimate_count == 1:
                    print(f"已估计 {estimate_count}/{successful_parses} 个查询...")

            except Exception as e:
                print(f"错误: 估计查询 {line_id} 时出错: {str(e)}")
                f.write('NaN\n')
                latencies.append(0.0)
                estimate_failures += 1
                results.append({
                    'query_id': line_id,
                    'sql': sql,
                    'estimate': None,
                    'error': str(e)
                })

    # 保存每子查询的估计时间
    bench_display = 'STATS' if benchmark == 'Stats' else benchmark
    time_file = os.path.join(checkpoint_dir, f"estimate_time_{bench_display}.txt")
    with open(time_file, 'w') as tf:
        for lat in latencies:
            tf.write(f"{lat}\n")

    total_time = time.time() - start_time

    print(f"\n评估完成！")
    print(f"预解析耗时（不计入估计时间）: {parse_time:.2f} 秒")
    print(f"估计总时间: {total_time:.2f} 秒")
    print(f"平均每个查询: {total_time/len(results):.4f} 秒" if len(results) > 0 else "")
    print(f"结果已保存到: {output_file}")
    print(f"成功估计的查询数: {sum(1 for r in results if r.get('estimate') is not None)}/{successful_parses}")
    print(f"解析失败的查询数: {parse_failures}")
    print(f"估计失败的查询数: {estimate_failures}")

    # 返回摘要信息
    return {
        'benchmark': benchmark,
        'total_time': total_time,
        'parse_time': parse_time,
        'num_queries': len(results),
        'successful_queries': sum(1 for r in results if r.get('estimate') is not None),
        'parse_failures': parse_failures,
        'estimate_failures': estimate_failures,
        'output_file': output_file,
        'time_file': time_file
    }

class RawSqlValue:
    def __init__(self, value):
        self.value = value

    def __str__(self):
        return self.value

    def __eq__(self, other):
        if isinstance(other, RawSqlValue):
            return self.value == other.value
        if isinstance(other, str):
            return self.value == other
        return NotImplemented

    def __lt__(self, other):
        if isinstance(other, RawSqlValue):
            return self.value < other.value
        if isinstance(other, str):
            return self.value < other
        return NotImplemented

    def __le__(self, other):
        if isinstance(other, RawSqlValue):
            return self.value <= other.value
        if isinstance(other, str):
            return self.value <= other
        return NotImplemented

    def __gt__(self, other):
        if isinstance(other, RawSqlValue):
            return self.value > other.value
        if isinstance(other, str):
            return self.value > other
        return NotImplemented

    def __ge__(self, other):
        if isinstance(other, RawSqlValue):
            return self.value >= other.value
        if isinstance(other, str):
            return self.value >= other
        return NotImplemented


def parse_sql_value(raw_value, keep_type_cast=True):
    raw_value = raw_value.strip()
    if raw_value.upper() == "NULL":
        return RawSqlValue("NULL")
    if raw_value.startswith("'"):
        if "::" in raw_value:
            if keep_type_cast:
                return RawSqlValue(raw_value)
            value_part = raw_value.split("::", 1)[0].strip()
            if value_part.endswith("'") and len(value_part) >= 2:
                return value_part[1:-1]
            return value_part
        if raw_value.endswith("'") and len(raw_value) >= 2:
            return raw_value[1:-1]
        return RawSqlValue(raw_value)
    if re.match(r"^-?\d+$", raw_value):
        return int(raw_value)
    if re.match(r"^-?\d+\.\d+$", raw_value):
        return float(raw_value)
    return RawSqlValue(raw_value)


def sql_to_joingraph(sql_query, keep_type_cast=True):
    """
    解析SQL查询并转换为JoinQueryGraph（用于 Stats benchmark）

    参数:
        sql_query: SQL查询字符串
        keep_type_cast: 是否保留类似 ::timestamp 的类型标注

    返回:
        JoinQueryGraph实例
    """
    import re
    query = JoinQueryGraph()

    # 解析FROM子句获取表别名
    from_match = re.search(r'FROM\s+(.+?)(?:\s+WHERE|$)', sql_query, re.IGNORECASE)
    if not from_match:
        raise ValueError("Invalid SQL query: missing FROM clause")

    # 解析表别名
    tables = [t.strip() for t in from_match.group(1).split(',')]
    for table in tables:
        if ' AS ' in table.upper():
            table_name, alias = re.split(r'\s+AS\s+', table, flags=re.IGNORECASE)
        else:
            table_name = alias = table.split()[-1]
        query.addAlias(table_name.strip(), alias.strip())

    # 解析WHERE子句获取连接条件和过滤谓词
    where_match = re.search(r'WHERE\s+(.+?)(?:\s*;|$)', sql_query, re.IGNORECASE)
    if where_match:
        join_conditions = re.split(r'\s+AND\s+', where_match.group(1), flags=re.IGNORECASE)

        for condition in join_conditions:
            condition = condition.strip().rstrip(';')
            # 解析形如 table1.column1=table2.column2 的条件
            match = re.match(r'(\w+)\.(\w+)\s*=\s*(\w+)\.(\w+)$', condition, flags=re.IGNORECASE)
            if match:
                table1, col1, table2, col2 = match.groups()
                query.addJoin(table1, col1, table2, col2)
                continue

            pred_match = re.match(r'(\w+)\.(\w+)\s*(=|<=|>=|<|>)\s*(.+)$', condition, flags=re.IGNORECASE)
            if pred_match:
                alias, col, op, raw_value = pred_match.groups()
                value = parse_sql_value(raw_value, keep_type_cast=keep_type_cast)
                query.addPredicate(alias, col, op, value)

    return query


def collect_predicate_examples(sqls, max_per_type=2):
    import re
    examples = {}
    for idx, sql in enumerate(sqls, 1):
        where_match = re.search(r'WHERE\s+(.+?)(?:\s*;|$)', sql, re.IGNORECASE)
        if not where_match:
            continue
        conditions = re.split(r'\s+AND\s+', where_match.group(1), flags=re.IGNORECASE)
        for condition in conditions:
            condition = condition.strip().rstrip(';')
            join_match = re.match(r'(\w+)\.(\w+)\s*=\s*(\w+)\.(\w+)$', condition, flags=re.IGNORECASE)
            if join_match:
                continue
            pred_match = re.match(r'(\w+)\.(\w+)\s*(=|<=|>=|<|>)\s*(.+)$', condition, flags=re.IGNORECASE)
            if not pred_match:
                continue
            _, _, op, raw_value = pred_match.groups()
            raw_value = raw_value.strip()
            value_type = 'timestamp' if '::' in raw_value else 'numeric' if re.match(r'^-?\d+(\.\d+)?$', raw_value) else 'other'
            key = f"{op}:{value_type}"
            if key not in examples:
                examples[key] = []
            if len(examples[key]) < max_per_type:
                examples[key].append(idx)
    return examples


def print_sql_roundtrip(sqls, indices):
    for idx in indices:
        if idx < 1 or idx > len(sqls):
            print(f"索引超出范围: {idx}")
            continue
        sql = sqls[idx - 1].strip()
        print(f"\n--- SQL #{idx} 原始 ---")
        print(sql)
        try:
            query = sql_to_joingraph(sql, keep_type_cast=True)
            query.buildJoinGraph()
            print("--- getSQLQuery ---")
            print(query.getSQLQuery())
        except Exception as e:
            print(f"解析失败: {e}")

print("评估函数已定义，可以使用 evaluate_safebound_benchmark() 函数评估各个 benchmark")

评估函数已定义，可以使用 evaluate_safebound_benchmark() 函数评估各个 benchmark


### Stats Benchmark

In [ ]:
# ==================== 配置区域 ====================
# Stats Benchmark 的子查询文件路径
# 请根据实际情况修改此路径
# stats_sql_file = os.path.join(rootFileDirectory, 'subqueries-stats.sql')
# 如果文件不在 SafeBound 目录下，可以使用绝对路径，例如：
stats_sql_file = os.path.abspath('../Benchmark/workloads/STATS-CEB/subquery/subquery.sql')
# ================================================

# 统计对象路径（从 Build 阶段生成）
stats_stat_object = os.path.join(checkpoint_dir, 'SafeBound_3_Stats.pkl')

# 检查文件是否存在
if not os.path.exists(stats_stat_object):
    print(f"错误: 统计对象文件不存在: {stats_stat_object}")
    print("请先运行 Build 阶段的 Stats Benchmark")
elif not os.path.exists(stats_sql_file):
    print(f"错误: SQL 文件不存在: {stats_sql_file}")
    print("请检查并修改 stats_sql_file 路径")
else:
    # 执行评估
    stats_results = evaluate_safebound_benchmark(
        benchmark='Stats',
        sql_file_path=stats_sql_file,
        stat_object_path=stats_stat_object
    )
    stats_results


加载统计对象: /home/liwei/starCE/experiment/checkpoint/SafeBound/SafeBound_3_Stats.pkl
统计对象大小: 1406395 字节

读取 SQL 查询文件: /home/liwei/starCE/Benchmark/workloads/STATS-CEB/subquery/subquery.sql
查询数量: 2471

预解析 2471 条 SQL 查询...
  预解析进度: 247/2471
  预解析进度: 494/2471
  预解析进度: 741/2471
  预解析进度: 988/2471
  预解析进度: 1235/2471
  预解析进度: 1482/2471
  预解析进度: 1729/2471
  预解析进度: 1976/2471
  预解析进度: 2223/2471
  预解析进度: 2470/2471
预解析完成: 成功 2471, 失败 0, 耗时 0.71s（不计入估计时间）
已估计 1/2471 个查询...
已估计 247/2471 个查询...
已估计 494/2471 个查询...
已估计 741/2471 个查询...
已估计 988/2471 个查询...
已估计 1235/2471 个查询...
已估计 1482/2471 个查询...
已估计 1729/2471 个查询...
已估计 1976/2471 个查询...
已估计 2223/2471 个查询...
已估计 2470/2471 个查询...

评估完成！
预解析耗时（不计入估计时间）: 0.71 秒
估计总时间: 2.46 秒
平均每个查询: 0.0010 秒
结果已保存到: /home/liwei/starCE/experiment/checkpoint/SafeBound/SafeBound_3_Stats_evaluate_results.txt
成功估计的查询数: 2471/2471
解析失败的查询数: 0
估计失败的查询数: 0


### JOBLight Benchmark

In [ ]:
# ==================== 配置区域 ====================
# JOBLight Benchmark 的子查询文件路径
# 请根据实际情况修改此路径
# 如果文件不在 SafeBound 目录下，可以使用绝对路径，例如：
joblight_sql_file = os.path.abspath('../Benchmark/workloads/JOBLight/subquery/subquery2.sql')
# ================================================

# 统计对象路径（从 Build 阶段生成）
joblight_stat_object = os.path.join(checkpoint_dir, 'SafeBound_3_JOBLight.pkl')

# 检查文件是否存在
if not os.path.exists(joblight_stat_object):
    print(f"错误: 统计对象文件不存在: {joblight_stat_object}")
    print("请先运行 Build 阶段的 JOBLight Benchmark")
elif not os.path.exists(joblight_sql_file):
    print(f"错误: SQL 文件不存在: {joblight_sql_file}")
    print("请检查并修改 joblight_sql_file 路径")
else:
    # 执行评估
    joblight_results = evaluate_safebound_benchmark(
        benchmark='JOBLight',
        sql_file_path=joblight_sql_file,
        stat_object_path=joblight_stat_object
    )
    joblight_results


加载统计对象: /home/liwei/starCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBLight.pkl
统计对象大小: 378926 字节

读取 SQL 查询文件: /home/liwei/starCE/Benchmark/workloads/JOBLight/subquery/subquery2.sql
查询数量: 451

预解析 451 条 SQL 查询...
  预解析进度: 45/451
  预解析进度: 90/451
  预解析进度: 135/451
  预解析进度: 180/451
  预解析进度: 225/451
  预解析进度: 270/451
  预解析进度: 315/451
  预解析进度: 360/451
  预解析进度: 405/451
  预解析进度: 450/451
预解析完成: 成功 451, 失败 0, 耗时 4.03s（不计入估计时间）
已估计 1/451 个查询...
已估计 45/451 个查询...
已估计 90/451 个查询...
已估计 135/451 个查询...
已估计 180/451 个查询...
已估计 225/451 个查询...
已估计 270/451 个查询...
已估计 315/451 个查询...
已估计 360/451 个查询...
已估计 405/451 个查询...
已估计 450/451 个查询...

评估完成！
预解析耗时（不计入估计时间）: 4.03 秒
估计总时间: 0.39 秒
平均每个查询: 0.0009 秒
结果已保存到: /home/liwei/starCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBLight_evaluate_results.txt
成功估计的查询数: 451/451
解析失败的查询数: 0
估计失败的查询数: 0


### JOBLightRanges Benchmark

In [ ]:
# ==================== 配置区域 ====================
# JOBLightRanges Benchmark 的子查询文件路径
# 请根据实际情况修改此路径
# 如果文件不在 SafeBound 目录下，可以使用绝对路径，例如：
joblightranges_sql_file = os.path.abspath('../Benchmark/workloads/JOBLightRanges/subquery/subquery2.sql')
# ================================================

# 统计对象路径（从 Build 阶段生成）
joblightranges_stat_object = os.path.join(checkpoint_dir, 'SafeBound_3_JOBLightRanges.pkl')

# 检查文件是否存在
if not os.path.exists(joblightranges_stat_object):
    print(f"错误: 统计对象文件不存在: {joblightranges_stat_object}")
    print("请先运行 Build 阶段的 JOBLightRanges Benchmark")
elif not os.path.exists(joblightranges_sql_file):
    print(f"错误: SQL 文件不存在: {joblightranges_sql_file}")
    print("请检查并修改 joblightranges_sql_file 路径")
else:
    # 执行评估
    joblightranges_results = evaluate_safebound_benchmark(
        benchmark='JOBLightRanges',
        sql_file_path=joblightranges_sql_file,
        stat_object_path=joblightranges_stat_object
    )
    joblightranges_results


加载统计对象: /home/liwei/starCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBLightRanges.pkl
统计对象大小: 628524 字节

读取 SQL 查询文件: /home/liwei/starCE/Benchmark/workloads/JOBLightRanges/subquery/subquery2.sql
查询数量: 8292

预解析 8292 条 SQL 查询...
  预解析进度: 829/8292
  预解析进度: 1658/8292
  预解析进度: 2487/8292
  预解析进度: 3316/8292
  预解析进度: 4145/8292
  预解析进度: 4974/8292
  预解析进度: 5803/8292
  预解析进度: 6632/8292
  预解析进度: 7461/8292
  预解析进度: 8290/8292
预解析完成: 成功 8292, 失败 0, 耗时 91.93s（不计入估计时间）
已估计 1/8292 个查询...
已估计 829/8292 个查询...
已估计 1658/8292 个查询...
已估计 2487/8292 个查询...
已估计 3316/8292 个查询...
已估计 4145/8292 个查询...
已估计 4974/8292 个查询...
已估计 5803/8292 个查询...
已估计 6632/8292 个查询...
已估计 7461/8292 个查询...
已估计 8290/8292 个查询...

评估完成！
预解析耗时（不计入估计时间）: 91.93 秒
估计总时间: 9.02 秒
平均每个查询: 0.0011 秒
结果已保存到: /home/liwei/starCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBLightRanges_evaluate_results.txt
成功估计的查询数: 8292/8292
解析失败的查询数: 0
估计失败的查询数: 0


### JOBM Benchmark

In [ ]:
# ==================== 配置区域 ====================
# JOBM Benchmark 的子查询文件路径
# 请根据实际情况修改此路径
# jobm_sql_file = os.path.join(rootFileDirectory, 'subqueries.sql')
# 如果文件不在 SafeBound 目录下，可以使用绝对路径，例如：
jobm_sql_file = os.path.abspath('../Benchmark/workloads/JOBM/subquery/subquery2.sql')
# ================================================

# 统计对象路径（从 Build 阶段生成）
jobm_stat_object = os.path.join(checkpoint_dir, 'SafeBound_3_JOBM.pkl')

# 检查文件是否存在
if not os.path.exists(jobm_stat_object):
    print(f"错误: 统计对象文件不存在: {jobm_stat_object}")
    print("请先运行 Build 阶段的 JOBM Benchmark")
elif not os.path.exists(jobm_sql_file):
    print(f"错误: SQL 文件不存在: {jobm_sql_file}")
    print("请检查并修改 jobm_sql_file 路径")
else:
    # 执行评估
    jobm_results = evaluate_safebound_benchmark(
        benchmark='JOBM',
        sql_file_path=jobm_sql_file,
        stat_object_path=jobm_stat_object
    )
    jobm_results


加载统计对象: /home/liwei/starCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBM.pkl
统计对象大小: 1274617 字节

读取 SQL 查询文件: /home/liwei/starCE/Benchmark/workloads/JOBM/subquery/subquery2.sql
查询数量: 6424

预解析 6424 条 SQL 查询...
  预解析进度: 642/6424
  预解析进度: 1284/6424
  预解析进度: 1926/6424
  预解析进度: 2568/6424
  预解析进度: 3210/6424
  预解析进度: 3852/6424
  预解析进度: 4494/6424
  预解析进度: 5136/6424
  预解析进度: 5778/6424
  预解析进度: 6420/6424
预解析完成: 成功 6424, 失败 0, 耗时 131.30s（不计入估计时间）
已估计 1/6424 个查询...
已估计 642/6424 个查询...
已估计 1284/6424 个查询...
已估计 1926/6424 个查询...
已估计 2568/6424 个查询...
已估计 3210/6424 个查询...
已估计 3852/6424 个查询...
已估计 4494/6424 个查询...
已估计 5136/6424 个查询...
已估计 5778/6424 个查询...
已估计 6420/6424 个查询...

评估完成！
预解析耗时（不计入估计时间）: 131.30 秒
估计总时间: 66.98 秒
平均每个查询: 0.0104 秒
结果已保存到: /home/liwei/starCE/experiment/checkpoint/SafeBound/SafeBound_3_JOBM_evaluate_results.txt
成功估计的查询数: 6424/6424
解析失败的查询数: 0
估计失败的查询数: 0


### StatsJoin Benchmark

In [ ]:
# ==================== 配置区域 ====================
# StatsJoin Benchmark 的子查询文件路径（复用 Stats 的统计对象 SafeBound_3_Stats.pkl）
statsjoin_sql_file = os.path.abspath('../Benchmark/workloads/StatsJoin/subquery/subquery.sql')
# ================================================

# 统计对象路径（从 Build 阶段生成，复用 Stats 统计对象）
statsjoin_stat_object = os.path.join(checkpoint_dir, 'SafeBound_3_Stats.pkl')

# 检查文件是否存在
if not os.path.exists(statsjoin_stat_object):
    print(f"错误: 统计对象文件不存在: {statsjoin_stat_object}")
    print("请先运行 Build 阶段的 Stats Benchmark")
elif not os.path.exists(statsjoin_sql_file):
    print(f"错误: SQL 文件不存在: {statsjoin_sql_file}")
    print("请检查并修改 statsjoin_sql_file 路径")
else:
    # 执行评估
    statsjoin_results = evaluate_safebound_benchmark(
        benchmark='StatsJoin',
        sql_file_path=statsjoin_sql_file,
        stat_object_path=statsjoin_stat_object
    )
    statsjoin_results

In [ ]:
# 将各个Benchmark的评估时间和构建时间输出为CSV文件
import csv
import os

# 完整的基准测试列表
all_benchmarks = {
    'Stats': {'BuildTime': None, 'EvaluationTime': None, 'StatisticsSize': None, 'ParseTime': None},
    'JOBLight': {'BuildTime': None, 'EvaluationTime': None, 'StatisticsSize': None, 'ParseTime': None},
    'JOBLightRanges': {'BuildTime': None, 'EvaluationTime': None, 'StatisticsSize': None, 'ParseTime': None},
    'JOBM': {'BuildTime': None, 'EvaluationTime': None, 'StatisticsSize': None, 'ParseTime': None},
    'StatsJoin': {'BuildTime': None, 'EvaluationTime': None, 'StatisticsSize': None, 'ParseTime': None}
}

# 从变量中获取评估时间（如果已运行）
if 'stats_results' in locals() and stats_results:
    if isinstance(stats_results, dict) and 'total_time' in stats_results:
        all_benchmarks['Stats']['EvaluationTime'] = stats_results['total_time']
        all_benchmarks['Stats']['ParseTime'] = stats_results.get('parse_time')

if 'joblight_results' in locals() and joblight_results:
    if isinstance(joblight_results, dict) and 'total_time' in joblight_results:
        all_benchmarks['JOBLight']['EvaluationTime'] = joblight_results['total_time']
        all_benchmarks['JOBLight']['ParseTime'] = joblight_results.get('parse_time')

if 'joblightranges_results' in locals() and joblightranges_results:
    if isinstance(joblightranges_results, dict) and 'total_time' in joblightranges_results:
        all_benchmarks['JOBLightRanges']['EvaluationTime'] = joblightranges_results['total_time']
        all_benchmarks['JOBLightRanges']['ParseTime'] = joblightranges_results.get('parse_time')

if 'jobm_results' in locals() and jobm_results:
    if isinstance(jobm_results, dict) and 'total_time' in jobm_results:
        all_benchmarks['JOBM']['EvaluationTime'] = jobm_results['total_time']
        all_benchmarks['JOBM']['ParseTime'] = jobm_results.get('parse_time')

if 'statsjoin_results' in locals() and statsjoin_results:
    if isinstance(statsjoin_results, dict) and 'total_time' in statsjoin_results:
        all_benchmarks['StatsJoin']['EvaluationTime'] = statsjoin_results['total_time']
        all_benchmarks['StatsJoin']['ParseTime'] = statsjoin_results.get('parse_time')

# 从变量中获取构建时间（如果已运行并保存到变量）
# build_safebound_benchmark 返回 (time, size)，已保存为 stats_time, joblight_time 等变量
build_time_vars = {
    'Stats': 'stats_time',
    'JOBLight': 'joblight_time',
    'JOBLightRanges': 'joblightranges_time',
    'JOBM': 'jobm_time',
    'StatsJoin': 'statsjoin_time'
}

for benchmark, var_name in build_time_vars.items():
    if var_name in locals():
        var_value = locals()[var_name]
        # 如果是数字，直接使用
        if isinstance(var_value, (int, float)):
            all_benchmarks[benchmark]['BuildTime'] = var_value

# 优先从变量中获取统计对象大小
# build_safebound_benchmark 返回 (time, size)，已保存为 stats_size, joblight_size 等变量
size_vars = {
    'Stats': 'stats_size',
    'JOBLight': 'joblight_size',
    'JOBLightRanges': 'joblightranges_size',
    'JOBM': 'jobm_size',
    'StatsJoin': 'statsjoin_size'
}

for benchmark, var_name in size_vars.items():
    if var_name in locals():
        var_value = locals()[var_name]
        if isinstance(var_value, (int, float)):
            all_benchmarks[benchmark]['StatisticsSize'] = var_value

# 若变量不存在，再从统计对象文件路径获取大小
stat_object_paths = {
    benchmark: os.path.join(checkpoint_dir, f"SafeBound_3_{benchmark}.pkl")
    for benchmark in all_benchmarks.keys()
}

for benchmark, stat_path in stat_object_paths.items():
    if all_benchmarks[benchmark]['StatisticsSize'] is None and os.path.exists(stat_path):
        try:
            all_benchmarks[benchmark]['StatisticsSize'] = os.path.getsize(stat_path)
        except OSError:
            all_benchmarks[benchmark]['StatisticsSize'] = None

print("正在收集评估时间、构建时间和统计对象大小数据...")

# 生成 CSV 文件
csv_file_path = os.path.join(checkpoint_dir, 'benchmark_times.csv')

# 准备 CSV 数据
csv_data = []
for benchmark_name, times in all_benchmarks.items():
    build_time = times['BuildTime']
    eval_time = times['EvaluationTime']
    stats_size = times['StatisticsSize']
    parse_time = times['ParseTime']
    csv_data.append({
        'Benchmark': benchmark_name,
        'BuildTime': build_time if build_time is not None else '',
        'StatisticsSize': stats_size if stats_size is not None else '',
        'EvaluationTime': eval_time if eval_time is not None else '',
        'ParseTime': parse_time if parse_time is not None else ''
    })

# 写入 CSV 文件
with open(csv_file_path, 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['Benchmark', 'BuildTime', 'StatisticsSize', 'EvaluationTime', 'ParseTime']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(csv_data)

print(f"\n数据已保存到: {csv_file_path}")
print("\nCSV 内容预览：")
print(f"{'Benchmark':<20} {'BuildTime':<15} {'StatisticsSize':<18} {'EvaluationTime':<15} {'ParseTime':<12}")
print("-" * 85)
for row in csv_data:
    build_time = str(row['BuildTime']) if row['BuildTime'] else 'N/A'
    stats_size = str(row['StatisticsSize']) if row['StatisticsSize'] else 'N/A'
    eval_time = str(row['EvaluationTime']) if row['EvaluationTime'] else 'N/A'
    parse_t = str(row.get('ParseTime', '')) if row.get('ParseTime', '') else 'N/A'
    print(f"{row['Benchmark']:<20} {build_time:<15} {stats_size:<18} {eval_time:<15} {parse_t:<12}")


正在收集评估时间、构建时间和统计对象大小数据...

数据已保存到: /home/liwei/starCE/experiment/checkpoint/SafeBound/benchmark_times.csv

CSV 内容预览：
Benchmark            BuildTime       StatisticsSize     EvaluationTime  ParseTime   
-------------------------------------------------------------------------------------
Stats                17.3802         1268042.0          2.4570987224578857 0.711951494216919
JOBLight             209.419419      362156.0           0.3905477523803711 4.034812688827515
JOBLightRanges       373.044615      608152.0           9.024212837219238 91.93029570579529
JOBM                 1893.179702     3567828.0          66.98474311828613 131.29909896850586
